In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

[ RAG 구현 절차 ]

```
1.	문서의 내용을 읽는다(document_loader를 이용)
(1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
(2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
%pip install --upgrade --quiet  docx2txt
2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
(1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
%pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
(1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
(2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
%pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
(1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
%pip install –q langchain langchainhub
http://smith.langchain.com에서 key생성 .env key(LANGCHAIN_API_KEY) 추가

```

# 2. 문서를 쪼개면서 읽기(o)

In [ ]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(  # 문서를 쪼개는 기준이 문자수
    chunk_size=1500, #문서를 쪼갤때 1500글자씩 쪼개
    chunk_overlap=200
)
# 1번째 chunk 1~1450글자
# 2번째 chunk 1250~1750글자
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서 쪼개면서 읽는 시간 :', runtime)

# 3. 쪼갠문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : upstage의 text-embedding-3-large (기본:text-embedding-ada-002)
- 벡터 데이터베이스 : chroma

In [ ]:
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model="embedding-query"
)

In [ ]:
doc_result = embeddings.embed_documents(
    ["소득세법 어쩌구 저쩌구", documents[0].page_content]
)
print(len(doc_result), len(doc_result[0]), len(doc_result[1]))

In [ ]:
len(embeddings), len(embeddings[0]), len(embeddings[1])

In [ ]:
len(embedding.embed_query("소득세"))

In [ ]:
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때
# database = Chroma.from_documents(                                 
#     documents=documents,
#     embedding=embeddings,
#     collection_name="tax-collection", # 생략시 이름 랜덤
#     persist_directory='chroma_upstage'      # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db날라감
# )
# 이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function=embeddings,
    collection_name="tax-collection",
    persist_directory='chroma_upstage'
)

# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [ ]:
retrieved_docs

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [ ]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문자입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요
[context]는 다음과 같아요
{retrieved_docs}
Question : {query}"""

In [ ]:
ai_message = qa_chain.invoke(prompt)

In [ ]:
print(ai_message.content)

# 5. Augmentation을 위한 제공되는 Prompt활용하여 langchain으로 답변 생성

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
```
 query -> retriever전달(백터 검색 수행) 
 -> retrieval문서 -> prompt의 {context}에 삽입
 -> query -> prompt의 {question}에 삽입
```

In [ ]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [ ]:
ai_message = qa_chain.invoke({"query":query})

In [ ]:
ai_message